# Lab 05: Moment Generating Functions

*SF250 Probability Theory and Statistics. Written in C, run from this notebook.*


# Moment Generating Functions in C

Two programs that compute an MGF two ways and check they agree:

1. **Closed form**, the textbook formula.
2. **Brute force**, straight from the definition `M(t) = E[e^{tX}]`
   (a sum for discrete, an integral for continuous).

If both columns match, the formula is right. That is the whole experiment.

## Build and run

```
make          # builds both programs
make run      # runs one example of each (make run-discrete / run-continuous for one)
make run-discrete ARGS_D="poisson 4"        # your own arguments
make run-continuous ARGS_C="uniform 0 2"
make test     # one small example per distribution
make big      # stress test: n = 1e9, lambda = 1e11, 2e7 integration steps
make clean
```

## What is an MGF?

`M(t) = E[e^{tX}]`. It packs every moment of X into one function:

- `M(0) = 1` always.
- `M'(0) = E[X]` (the mean).
- `M''(0) = E[X^2]`, so `Var(X) = M''(0) - M'(0)^2`.

Better for computing, take `K(t) = log M(t)` (the cumulant function):

- `K'(0) = mean`
- `K''(0) = variance`

The programs use K because for big n, `M(t)` is astronomically large. With
n = 1e9, `M(1)` is about `e^(3.6e8)`, which overflows a double, but `K(1)` is just 3.6e8.

## Program 1: `discrete_moment_generating_function`

| Distribution | Usage | M(t) | mean | variance |
|---|---|---|---|---|
| Bernoulli(p) | `./discrete_moment_generating_function bernoulli 0.3` | `1-p+p·e^t` | p | p(1-p) |
| Binomial(n,p) | `./discrete_moment_generating_function binomial 20 0.4` | `(1-p+p·e^t)^n` | np | np(1-p) |
| Poisson(λ) | `./discrete_moment_generating_function poisson 4` | `exp(λ(e^t-1))` | λ | λ |

Notes:
- `n` is a `size_t`, so it can be as large as about 1.8e19.
- Bernoulli is just Binomial with n = 1.
- The brute-force sum only visits k within 12 standard deviations of the
  mean, so n = 1e9 takes under a second.
- Expect a small error at huge n (about 5e-7 at n = 1e9). `lgamma` loses
  precision when its argument is that large. It is not a bug.

## Program 2: `continuous_moment_generating_function`

| Distribution | Usage | M(t) | mean | variance |
|---|---|---|---|---|
| Uniform(a,b) | `./continuous_moment_generating_function uniform 0 2` | `(e^{tb}-e^{ta}) / (t(b-a))` | (a+b)/2 | (b-a)²/12 |
| Exponential(λ) | `./continuous_moment_generating_function exponential 2` | `λ/(λ-t)`, only for `t<λ` | 1/λ | 1/λ² |
| Gaussian(μ,σ) | `./continuous_moment_generating_function gaussian 1 2` | `exp(μt + σ²t²/2)` | μ | σ² |

Add a last number to set the Simpson integration steps, e.g.
`./continuous_moment_generating_function gaussian 1 2 100000000`. More steps means a smaller error
but a slower run. The default is 1,000,000.

## Reading the output

```
         t         K closed      K brute-sum       |diff|   (K = log M)
     -1.00       -5.8297387       -5.8297387            0
      ...
moments from derivatives of K at t=0 ...
  mean  numeric 8.0000002  theory 8
  var   numeric 4.7999998  theory 4.8
```

- **K closed** is the formula. **K brute-sum** (or **K integral**) is the
  direct computation. **|diff|** is their gap and should be tiny.
- **mean / var numeric** are `K'(0)` and `K''(0)` estimated with finite
  differences. They should match the theory column to several digits.
- The two ASCII graphs are the pmf/pdf, then `M(t)`. If `M(t)` is too steep
  to draw, the program plots `log M(t)` instead and says so.

## Things to try

1. `./discrete_moment_generating_function binomial 20 0.4`: mean 8, variance 4.8. Check the graphs.
2. `./discrete_moment_generating_function poisson 4` vs `poisson 400`. The pmf turns into a bell
   curve, and `log M` stays a clean curve.
3. `./discrete_moment_generating_function binomial 1000 0.5` next to `./continuous_moment_generating_function gaussian 500 15.8`.
   The binomial is close to a Gaussian, so their `K(t)` agree near t = 0.
4. `./continuous_moment_generating_function exponential 2`. The table's t values stop short of
   λ because M(t) blows up as `t → λ`.
5. `./continuous_moment_generating_function uniform 0 2` vs `uniform -1 1`. Shifting the interval
   multiplies M(t) by `e^{tc}`, which shifts K by `tc`.


## Slide references

ชื่อย่อสไลด์ (เลขหลังชื่อคือเลขหน้า "Slide" มุมขวาล่าง):

| ย่อ | ชุดสไลด์ |
|---|---|
| **PF** | Probability Fundamentals |
| **DD** | Discrete Probability Distributions |
| **CD** | Continuous Distributions |
| **MGF** | Moment Generating Functions |
| **CM** | Combining and Manipulating Distributions |
| **TS** | Testing and Statistical Significance |

## 2. `moment_generating_functions`: Moment Generating Function

**คืออะไร:** คำนวณ MGF สองวิธีแล้วเทียบกัน วิธีแรกใช้สูตรสำเร็จ วิธีที่สองคำนวณจากนิยามตรงๆ (ผลรวมหรืออินทิกรัล) ถ้าตรงกันแปลว่าสูตรถูก
ดูรายละเอียดการรันที่ `moment_generating_functions/README.md`

**ทำงานยังไง:** MGF เก็บโมเมนต์ทั้งหมดไว้ในฟังก์ชันเดียว หาค่าเฉลี่ยและความแปรปรวนได้จากอนุพันธ์ที่จุดอ้างอิง (ทำให้ไม่ต้องคำนวณผลรวมหรืออินทิกรัลยาวๆ)

| ส่วนของโปรแกรม | อ้างอิงสไลด์ |
|---|---|
| ทำไมต้องมี MGF (คำนวณ E[X] ของ Binomial ตรงๆ ยาวและน่าเบื่อ) | MGF 2 |
| MGF แบบไม่ต่อเนื่อง, E[R] = g'(1), E[R²] = g''(1) + g'(1) | MGF 3 |
| Binomial: g(z) = (q + pz)ⁿ ได้ E = np, Var = npq | MGF 4 |
| ตาราง Bernoulli / Binomial / Poisson | MGF 5 |
| MGF แบบต่อเนื่อง g(s) = ∫e^(sx)·p(x)dx, E[X] = g'(0), E[X²] = g''(0) | MGF 8 |
| ตาราง Uniform / Exponential / Gaussian | MGF 9 |
| Var = E[X²] − E[X]² | DD 6 |

หมายเหตุสัญลักษณ์: สไลด์เขียน MGF แบบไม่ต่อเนื่องเป็น g(z) = Σ zʳ·p(r) แต่ lab ใช้ M(t) = E[e^(tX)] ซึ่งเป็นตัวเดียวกันเมื่อ z = eᵗ

⚠ **นอกสไลด์:** K(t) = log M(t) ที่ใช้เพื่อกัน overflow เมื่อ n ใหญ่มาก, การอินทิเกรตเชิงตัวเลขแบบ Simpson, และการหา mean/var ด้วย finite difference


## Running C in a notebook

This notebook uses the normal Python kernel. A cell that starts with `%%writefile` saves a C source file, and a cell that starts with `!` runs a shell command (`make`, then the compiled program). It needs `gcc` and `make`, which Google Colab already has. On a local Jupyter, install them first (for example `sudo apt install build-essential`). The C code is the same code as in the repository, written in 42 Norm style.


In [1]:
!mkdir -p moment_generating_functions

In [2]:
!gcc --version | head -1

gcc (GCC) 16.2.1 20261029 (Red Hat 16.2.1-3)


## Source code

One cell per file. Run them all to recreate the project.

In [3]:
%%writefile moment_generating_functions/plot.h
#ifndef PLOT_H
# define PLOT_H

# include <math.h>
# include <stdio.h>
# include <string.h>

# define PW 70
# define PH 16

typedef double	(*t_fn)(double, void *);

typedef struct s_curve
{
	const char	*title;
	t_fn		f;
	void		*ctx;
	double		lo;
	double		hi;
}	t_curve;

typedef struct s_plot
{
	double	y[PW];
	char	g[PH][PW + 1];
	double	ymin;
	double	ymax;
}	t_plot;

void	plot(t_curve *c);

#endif


Writing moment_generating_functions/plot.h


In [4]:
%%writefile moment_generating_functions/plot.c
#include "plot.h"

static void	sample(t_plot *p, t_curve *c)
{
	int	i;

	p->ymin = INFINITY;
	p->ymax = -INFINITY;
	i = -1;
	while (++i < PW)
	{
		p->y[i] = c->f(c->lo + (c->hi - c->lo) * i / (PW - 1), c->ctx);
		if (isfinite(p->y[i]))
		{
			p->ymin = fmin(p->ymin, p->y[i]);
			p->ymax = fmax(p->ymax, p->y[i]);
		}
	}
	if (!(p->ymax > p->ymin))
		p->ymax = p->ymin + 1;
}

static int	row_of(t_plot *p, int i)
{
	return ((int)((p->ymax - p->y[i]) / (p->ymax - p->ymin) * (PH - 1) + .5));
}

static void	fill(t_plot *p)
{
	int	r;
	int	i;

	memset(p->g, ' ', sizeof(p->g));
	r = -1;
	while (++r < PH)
		p->g[r][PW] = 0;
	i = -1;
	while (++i < PW)
		if (isfinite(p->y[i]))
			p->g[row_of(p, i)][i] = '*';
}

static void	show(t_plot *p, t_curve *c)
{
	int	r;

	printf("\n%s\n", c->title);
	r = -1;
	while (++r < PH)
		printf("%11.4g |%s\n", p->ymax - (p->ymax - p->ymin) * r / (PH - 1),
			p->g[r]);
	printf("%11s +", "");
	r = -1;
	while (++r < PW)
		putchar('-');
	printf("\n%11s  %-*.4g%.4g\n", "", PW - 8, c->lo, c->hi);
}

void	plot(t_curve *c)
{
	t_plot	p;

	sample(&p, c);
	fill(&p);
	show(&p, c);
}


Writing moment_generating_functions/plot.c


In [5]:
%%writefile moment_generating_functions/discrete.h
#ifndef DISCRETE_H
# define DISCRETE_H

# include <math.h>
# include <stdio.h>
# include <stdlib.h>
# include <string.h>
# include "plot.h"

typedef struct s_dist
{
	int		poisson;
	size_t	n;
	double	p;
	double	lam;
}	t_dist;

double	lae(double a, double b);
double	kclosed(t_dist *d, double t);
double	logpmf(t_dist *d, size_t k);
double	knum(t_dist *d, double t);
void	print_intro(t_dist *d);
void	print_table(t_dist *d);
void	print_moments(t_dist *d, double mean, double var);
void	plots(t_dist *d, double mean, double var);

#endif


Writing moment_generating_functions/discrete.h


In [6]:
%%writefile moment_generating_functions/discrete_math.c
#include "discrete.h"

double	lae(double a, double b)
{
	return (fmax(a, b) + log1p(exp(-fabs(a - b))));
}

double	kclosed(t_dist *d, double t)
{
	if (d->poisson)
		return (d->lam * expm1(t));
	return ((double)d->n * lae(log1p(-d->p), log(d->p) + t));
}

double	logpmf(t_dist *d, size_t k)
{
	double	lg;

	if (d->poisson)
		return (k * log(d->lam) - d->lam - lgamma(k + 1.0));
	lg = lgamma(d->n + 1.0) - lgamma(k + 1.0);
	lg = lg - lgamma((double)(d->n - k) + 1.0);
	return (lg + k * log(d->p) + (double)(d->n - k) * log1p(-d->p));
}


Writing moment_generating_functions/discrete_math.c


In [7]:
%%writefile moment_generating_functions/discrete_num.c
#include "discrete.h"

static void	tilt(t_dist *d, double t, double *mu, double *sd)
{
	double	q;

	if (d->poisson)
	{
		*mu = d->lam * exp(t);
		*sd = sqrt(*mu);
		return ;
	}
	q = exp(log(d->p) + t - lae(log1p(-d->p), log(d->p) + t));
	*mu = d->n * q;
	*sd = sqrt(d->n * q * (1 - q));
}

static void	bounds(t_dist *d, double t, size_t *lo, size_t *hi)
{
	double	mu;
	double	sd;
	double	fhi;

	tilt(d, t, &mu, &sd);
	*lo = (size_t)fmax(0, mu - 12 * sd - 5);
	fhi = mu + 12 * sd + 5;
	*hi = (size_t)fhi;
	if (!d->poisson && fhi > (double)d->n)
		*hi = d->n;
}

double	knum(t_dist *d, double t)
{
	size_t	r[2];
	size_t	k;
	double	m;
	double	s;

	bounds(d, t, &r[0], &r[1]);
	m = -INFINITY;
	k = r[0] - 1;
	while (++k <= r[1])
		m = fmax(m, logpmf(d, k) + t * k);
	s = 0;
	k = r[0] - 1;
	while (++k <= r[1])
		s += exp(logpmf(d, k) + t * k - m);
	return (m + log(s));
}


Writing moment_generating_functions/discrete_num.c


In [8]:
%%writefile moment_generating_functions/discrete_out.c
#include "discrete.h"

static const double	g_ts[7] = {-1, -.5, -.1, 0, .1, .5, .9};

void	print_intro(t_dist *d)
{
	const char	*name;

	if (d->poisson)
		printf("Poisson(lambda=%g)\n", d->lam);
	else
	{
		name = "Binomial";
		if (d->n == 1)
			name = "Bernoulli";
		printf("%s(n=%zu, p=%g)\n", name, d->n, d->p);
	}
	printf("M(t) = E[e^{tX}] = ");
	if (d->poisson)
		printf("exp(lambda (e^t - 1))\n");
	else
		printf("(1-p+p e^t)^n\n");
	printf("\n%8s %16s %16s %12s   (K = log M)\n", "t", "K closed",
		"K brute-sum", "|diff|");
}

void	print_table(t_dist *d)
{
	int		i;
	double	a;
	double	b;

	i = -1;
	while (++i < 7)
	{
		a = kclosed(d, g_ts[i]);
		b = knum(d, g_ts[i]);
		printf("%8.2f %16.8g %16.8g %12.3g\n", g_ts[i], a, b, fabs(a - b));
	}
}

void	print_moments(t_dist *d, double mean, double var)
{
	double	h;
	double	k1;
	double	k0;

	h = 1e-3;
	k1 = kclosed(d, h);
	k0 = kclosed(d, -h);
	printf("\nmoments from derivatives of K at t=0 ");
	printf("(K'(0)=mean, K''(0)=var):\n");
	printf("  mean  numeric %.8g  theory %.8g\n", (k1 - k0) / (2 * h), mean);
	printf("  var   numeric %.8g  theory %.8g\n", (k1 - 2 * kclosed(d, 0) + k0)
		/ (h * h), var);
}


Writing moment_generating_functions/discrete_out.c


In [9]:
%%writefile moment_generating_functions/discrete_cb.c
#include "discrete.h"

static double	cb_pmf(double x, void *ctx)
{
	return (exp(logpmf(ctx, (size_t)(x + .5))));
}

static double	cb_m(double t, void *ctx)
{
	return (exp(kclosed(ctx, t)));
}

static double	cb_k(double t, void *ctx)
{
	return (kclosed(ctx, t));
}

void	plots(t_dist *d, double mean, double var)
{
	t_curve	c;
	double	sd;

	sd = sqrt(var);
	c.ctx = d;
	c.title = "pmf P(X=k)";
	c.f = cb_pmf;
	c.lo = fmax(0, mean - 4 * sd);
	c.hi = mean + 4 * sd;
	if (c.hi - c.lo < 1)
		c.hi = c.lo + 1;
	plot(&c);
	c.lo = -2;
	c.hi = 2;
	c.title = "M(t)";
	c.f = cb_m;
	if (!(kclosed(d, 2) < 20))
	{
		c.title = "log M(t)  (M itself overflows / is too steep to draw)";
		c.f = cb_k;
	}
	plot(&c);
}


Writing moment_generating_functions/discrete_cb.c


In [10]:
%%writefile moment_generating_functions/discrete_moment_generating_function.c
#include "discrete.h"

static int	usage(char *prog)
{
	fprintf(stderr, "usage: %s bernoulli p | binomial n p | ", prog);
	fprintf(stderr, "poisson lambda\n");
	fprintf(stderr, "e.g.   %s binomial 1000000000 0.3\n", prog);
	return (1);
}

static int	check(t_dist *d)
{
	if (!d->poisson && !(d->p > 0 && d->p < 1))
	{
		fprintf(stderr, "need 0<p<1\n");
		return (1);
	}
	if (d->poisson && !(d->lam > 0))
	{
		fprintf(stderr, "need lambda>0\n");
		return (1);
	}
	return (0);
}

static int	parse(t_dist *d, int ac, char **av)
{
	char	*name;

	name = "";
	if (ac > 1)
		name = av[1];
	if (!strcmp(name, "bernoulli") && ac == 3)
	{
		d->n = 1;
		d->p = atof(av[2]);
	}
	else if (!strcmp(name, "binomial") && ac == 4)
	{
		d->n = strtoull(av[2], NULL, 10);
		d->p = atof(av[3]);
	}
	else if (!strcmp(name, "poisson") && ac == 3)
	{
		d->poisson = 1;
		d->lam = atof(av[2]);
	}
	else
		return (usage(av[0]));
	return (check(d));
}

int	main(int ac, char **av)
{
	t_dist	d;
	double	mean;
	double	var;

	memset(&d, 0, sizeof(d));
	if (parse(&d, ac, av))
		return (1);
	mean = d.lam;
	var = d.lam;
	if (!d.poisson)
	{
		mean = d.n * d.p;
		var = d.n * d.p * (1 - d.p);
	}
	print_intro(&d);
	print_table(&d);
	print_moments(&d, mean, var);
	plots(&d, mean, var);
	return (0);
}


Writing moment_generating_functions/discrete_moment_generating_function.c


In [11]:
%%writefile moment_generating_functions/continuous.h
#ifndef CONTINUOUS_H
# define CONTINUOUS_H

# include <math.h>
# include <stdio.h>
# include <stdlib.h>
# include <string.h>
# include "plot.h"

typedef struct s_cont
{
	char	kind;
	size_t	n;
	double	a;
	double	b;
	double	lam;
	double	mu;
	double	sg;
	double	mean;
	double	var;
	double	unit;
}	t_cont;

double	kclosed(t_cont *c, double t);
double	logpdf(t_cont *c, double x);
void	setup(t_cont *c);
int		check_params(t_cont *c);
double	knum(t_cont *c, double t);
void	print_title(t_cont *c);
void	print_intro(t_cont *c);
void	print_table(t_cont *c);
void	print_moments(t_cont *c);
void	plots(t_cont *c);

#endif


Writing moment_generating_functions/continuous.h


In [12]:
%%writefile moment_generating_functions/continuous_math.c
#include "continuous.h"

static double	lem(double x)
{
	if (x > 0)
	{
		if (x > 30)
			return (x + log1p(-exp(-x)) - log(x));
		return (log(expm1(x)) - log(x));
	}
	return (log(-expm1(x)) - log(-x));
}

double	kclosed(t_cont *c, double t)
{
	if (c->kind == 'u')
	{
		if (fabs(t * (c->b - c->a)) < 1e-12)
			return (t * (c->a + c->b) / 2);
		return (t * c->a + lem(t * (c->b - c->a)));
	}
	if (c->kind == 'e')
	{
		if (t < c->lam)
			return (log(c->lam) - log(c->lam - t));
		return (NAN);
	}
	return (c->mu * t + c->sg * c->sg * t * t / 2);
}

double	logpdf(t_cont *c, double x)
{
	double	z;

	if (c->kind == 'u')
	{
		if (x < c->a || x > c->b)
			return (-INFINITY);
		return (-log(c->b - c->a));
	}
	if (c->kind == 'e')
	{
		if (x < 0)
			return (-INFINITY);
		return (log(c->lam) - c->lam * x);
	}
	z = x - c->mu;
	return (-.5 * z * z / (c->sg * c->sg) - log(c->sg * sqrt(2 * M_PI)));
}

void	setup(t_cont *c)
{
	if (c->kind == 'u')
	{
		c->mean = (c->a + c->b) / 2;
		c->var = (c->b - c->a) * (c->b - c->a) / 12;
		c->unit = 1 / fmax(fabs(c->a) + fabs(c->b), 1e-9);
	}
	else if (c->kind == 'e')
	{
		c->mean = 1 / c->lam;
		c->var = 1 / (c->lam * c->lam);
		c->unit = c->lam;
	}
	else
	{
		c->mean = c->mu;
		c->var = c->sg * c->sg;
		c->unit = 1 / fmax(fabs(c->mu) + c->sg, 1e-9);
	}
}

int	check_params(t_cont *c)
{
	int	bad;

	bad = 0;
	if (c->kind == 'u' && !(c->b > c->a))
		bad = 1;
	if (c->kind == 'e' && !(c->lam > 0))
		bad = 1;
	if (c->kind == 'g' && !(c->sg > 0))
		bad = 1;
	if (bad)
		fprintf(stderr, "bad parameters\n");
	return (bad);
}


Writing moment_generating_functions/continuous_math.c


In [13]:
%%writefile moment_generating_functions/continuous_num.c
#include "continuous.h"

static int	range(t_cont *c, double t, double *lo, double *hi)
{
	if (c->kind == 'u')
	{
		*lo = c->a;
		*hi = c->b;
	}
	else if (c->kind == 'e')
	{
		if (t >= c->lam)
			return (0);
		*lo = 0;
		*hi = 60 / (c->lam - t);
	}
	else
	{
		*lo = c->mu + c->sg * c->sg * t - 12 * c->sg;
		*hi = c->mu + c->sg * c->sg * t + 12 * c->sg;
	}
	return (1);
}

static double	weight(size_t i, size_t n)
{
	if (i == 0 || i == n)
		return (1);
	if (i % 2)
		return (4);
	return (2);
}

static double	vmax(t_cont *c, double t, double *r)
{
	double	d;
	double	m;
	double	x;
	size_t	i;

	d = (r[1] - r[0]) / c->n;
	m = -INFINITY;
	i = -1;
	while (++i <= c->n)
	{
		x = r[0] + i * d;
		m = fmax(m, logpdf(c, x) + t * x);
	}
	return (m);
}

static double	vsum(t_cont *c, double t, double *r, double m)
{
	double	d;
	double	s;
	double	x;
	size_t	i;

	d = (r[1] - r[0]) / c->n;
	s = 0;
	i = -1;
	while (++i <= c->n)
	{
		x = r[0] + i * d;
		s += weight(i, c->n) * exp(logpdf(c, x) + t * x - m);
	}
	return (s * d / 3);
}

double	knum(t_cont *c, double t)
{
	double	r[2];
	double	m;

	if (!range(c, t, &r[0], &r[1]))
		return (NAN);
	m = vmax(c, t, r);
	return (m + log(vsum(c, t, r, m)));
}


Writing moment_generating_functions/continuous_num.c


In [14]:
%%writefile moment_generating_functions/continuous_out.c
#include "continuous.h"

static const double	g_ts[7] = {-1, -.5, -.1, 0, .1, .5, .9};

void	print_title(t_cont *c)
{
	if (c->kind == 'u')
	{
		printf("Uniform(a=%g, b=%g)\n", c->a, c->b);
		printf("M(t) = (e^{tb} - e^{ta}) / (t (b-a))\n");
	}
	else if (c->kind == 'e')
	{
		printf("Exponential(lambda=%g)\n", c->lam);
		printf("M(t) = lambda / (lambda - t),  t < lambda\n");
	}
	else
	{
		printf("Gaussian(mu=%g, sigma=%g)\n", c->mu, c->sg);
		printf("M(t) = exp(mu t + sigma^2 t^2 / 2)\n");
	}
}

void	print_intro(t_cont *c)
{
	printf("Simpson intervals N=%zu\n\n", c->n);
	printf("%10s %16s %16s %12s   (K = log M)\n", "t", "K closed", "K integral",
		"|diff|");
}

void	print_table(t_cont *c)
{
	int		i;
	double	t;
	double	x;
	double	y;

	i = -1;
	while (++i < 7)
	{
		t = g_ts[i] * c->unit;
		x = kclosed(c, t);
		y = knum(c, t);
		printf("%10.4g %16.8g %16.8g %12.3g\n", t, x, y, fabs(x - y));
	}
}

void	print_moments(t_cont *c)
{
	double	h;
	double	k1;
	double	k0;

	h = 1e-3 * c->unit;
	k1 = kclosed(c, h);
	k0 = kclosed(c, -h);
	printf("\nmoments from derivatives of K at t=0:\n");
	printf("  mean  numeric %.8g  theory %.8g\n", (k1 - k0) / (2 * h), c->mean);
	printf("  var   numeric %.8g  theory %.8g\n", (k1 - 2 * kclosed(c, 0) + k0)
		/ (h * h), c->var);
}


Writing moment_generating_functions/continuous_out.c


In [15]:
%%writefile moment_generating_functions/continuous_cb.c
#include "continuous.h"

static double	cb_pdf(double x, void *ctx)
{
	return (exp(logpdf(ctx, x)));
}

static double	cb_m(double t, void *ctx)
{
	return (exp(kclosed(ctx, t)));
}

static double	cb_k(double t, void *ctx)
{
	return (kclosed(ctx, t));
}

static void	pdf_range(t_cont *c, t_curve *cv)
{
	double	sd;

	sd = sqrt(c->var);
	if (c->kind == 'u')
	{
		cv->lo = c->a - .1 * (c->b - c->a);
		cv->hi = c->b + .1 * (c->b - c->a);
	}
	else if (c->kind == 'e')
	{
		cv->lo = 0;
		cv->hi = c->mean + 5 * sd;
	}
	else
	{
		cv->lo = c->mean - 4 * sd;
		cv->hi = c->mean + 4 * sd;
	}
}

void	plots(t_cont *c)
{
	t_curve	cv;

	cv.ctx = c;
	cv.title = "pdf f(x)";
	cv.f = cb_pdf;
	pdf_range(c, &cv);
	plot(&cv);
	cv.lo = -2 * c->unit;
	cv.hi = 2 * c->unit;
	if (c->kind == 'e')
		cv.hi = .95 * c->unit;
	cv.title = "M(t)";
	cv.f = cb_m;
	if (!(kclosed(c, cv.hi) < 20))
	{
		cv.title = "log M(t)";
		cv.f = cb_k;
	}
	plot(&cv);
}


Writing moment_generating_functions/continuous_cb.c


In [16]:
%%writefile moment_generating_functions/continuous_moment_generating_function.c
#include "continuous.h"

static int	usage(char *prog)
{
	fprintf(stderr, "usage: %s uniform a b [N] | exponential lambda [N] | ",
		prog);
	fprintf(stderr, "gaussian mu sigma [N]\n");
	fprintf(stderr, "e.g.   %s gaussian 1 2 100000000\n", prog);
	return (1);
}

static int	pick2(t_cont *c, char **av)
{
	if (!strcmp(av[1], "uniform"))
	{
		c->kind = 'u';
		c->a = atof(av[2]);
		c->b = atof(av[3]);
		return (4);
	}
	c->kind = 'g';
	c->mu = atof(av[2]);
	c->sg = atof(av[3]);
	return (4);
}

static int	pick(t_cont *c, int ac, char **av)
{
	char	*name;

	name = "";
	if (ac > 1)
		name = av[1];
	if (!strcmp(name, "exponential") && ac >= 3)
	{
		c->kind = 'e';
		c->lam = atof(av[2]);
		return (3);
	}
	if ((!strcmp(name, "uniform") || !strcmp(name, "gaussian")) && ac >= 4)
		return (pick2(c, av));
	return (0);
}

static int	parse(t_cont *c, int ac, char **av)
{
	int	na;

	na = pick(c, ac, av);
	if (!na)
		return (usage(av[0]));
	c->n = 1000000;
	if (ac > na)
		c->n = strtoull(av[na], NULL, 10);
	c->n += c->n & 1;
	if (!c->n)
		c->n = 2;
	return (check_params(c));
}

int	main(int ac, char **av)
{
	t_cont	c;

	memset(&c, 0, sizeof(c));
	if (parse(&c, ac, av))
		return (1);
	setup(&c);
	print_title(&c);
	print_intro(&c);
	print_table(&c);
	print_moments(&c);
	plots(&c);
	return (0);
}


Writing moment_generating_functions/continuous_moment_generating_function.c


In [17]:
%%writefile moment_generating_functions/Makefile
CFLAGS = -O2 -Wall -Wextra -std=c11 -D_GNU_SOURCE
DISC = discrete_moment_generating_function
CONT = continuous_moment_generating_function
DISC_SRCS = $(DISC).c discrete_math.c discrete_num.c discrete_out.c \
	discrete_cb.c plot.c
CONT_SRCS = $(CONT).c continuous_math.c continuous_num.c continuous_out.c \
	continuous_cb.c plot.c
ARGS_D = binomial 20 0.4
ARGS_C = gaussian 1 2

all: $(DISC) $(CONT)

$(DISC): $(DISC_SRCS) discrete.h plot.h
	$(CC) $(CFLAGS) $(DISC_SRCS) -o $@ -lm

$(CONT): $(CONT_SRCS) continuous.h plot.h
	$(CC) $(CFLAGS) $(CONT_SRCS) -o $@ -lm

# make run-discrete                          runs the discrete example
# make run-discrete ARGS_D="poisson 4"       runs with your own arguments
# make run-continuous ARGS_C="uniform 0 2"   same for the continuous program
run-discrete: $(DISC)
	./$(DISC) $(ARGS_D)

run-continuous: $(CONT)
	./$(CONT) $(ARGS_C)

run: run-discrete run-continuous

# every distribution, small numbers
test: all
	./$(DISC) bernoulli 0.3
	./$(DISC) binomial 20 0.4
	./$(DISC) poisson 4
	./$(CONT) uniform 0 2
	./$(CONT) exponential 2
	./$(CONT) gaussian 1 2

# stress: huge n / lambda / N
big: all
	./$(DISC) binomial 1000000000 0.3
	./$(DISC) poisson 100000000000
	./$(CONT) gaussian 1 2 20000000

clean:
	rm -f $(DISC) $(CONT)

re: clean all

.PHONY: all run run-discrete run-continuous test big clean re


Writing moment_generating_functions/Makefile


## Build

`make` compiles every `.c` file into one program.

In [18]:
!cd moment_generating_functions && make

cc -O2 -Wall -Wextra -std=c11 -D_GNU_SOURCE discrete_moment_generating_function.c discrete_math.c discrete_num.c discrete_out.c discrete_cb.c plot.c -o discrete_moment_generating_function -lm
cc -O2 -Wall -Wextra -std=c11 -D_GNU_SOURCE continuous_moment_generating_function.c continuous_math.c continuous_num.c continuous_out.c continuous_cb.c plot.c -o continuous_moment_generating_function -lm


## Run it

Binomial(20, 0.4): closed form vs brute-force sum.

In [19]:
!cd moment_generating_functions && ./discrete_moment_generating_function binomial 20 0.4

Binomial(n=20, p=0.4)
M(t) = E[e^{tX}] = (1-p+p e^t)^n

       t         K closed      K brute-sum       |diff|   (K = log M)
   -1.00       -5.8297387       -5.8297387            0
   -0.50       -3.4249675       -3.4249675     8.88e-16
   -0.10      -0.77616864      -0.77616864     1.11e-15
    0.00                0    6.6613381e-16     6.66e-16
    0.10       0.82415105       0.82415105        2e-15
    0.50        4.6141139        4.6141139            0
    0.90        9.1970613        9.1970613            0

moments from derivatives of K at t=0 (K'(0)=mean, K''(0)=var):
  mean  numeric 8.0000002  theory 8
  var   numeric 4.7999998  theory 4.8

pmf P(X=k)
     0.1797 |                               ****                                   
     0.1677 |                           ****                                       
     0.1557 |                                   *****                              
     0.1438 |                                                                   

Poisson(4):

In [20]:
!cd moment_generating_functions && ./discrete_moment_generating_function poisson 4

Poisson(lambda=4)
M(t) = E[e^{tX}] = exp(lambda (e^t - 1))

       t         K closed      K brute-sum       |diff|   (K = log M)
   -1.00       -2.5284822       -2.5284822     4.44e-16
   -0.50       -1.5738774       -1.5738774     2.22e-16
   -0.10      -0.38065033      -0.38065033     2.22e-16
    0.00                0                0            0
    0.10       0.42068367       0.42068367     5.55e-16
    0.50        2.5948851        2.5948851     8.88e-16
    0.90        5.8384124        5.8384124     8.88e-16

moments from derivatives of K at t=0 (K'(0)=mean, K''(0)=var):
  mean  numeric 4.0000007  theory 4
  var   numeric 4.0000003  theory 4

pmf P(X=k)
     0.1954 |               ***********                                            
     0.1824 |                                                                      
     0.1694 |                                                                      
     0.1564 |                          ******                                 

Gaussian(1, 2): closed form vs Simpson integral.

In [21]:
!cd moment_generating_functions && ./continuous_moment_generating_function gaussian 1 2

Gaussian(mu=1, sigma=2)
M(t) = exp(mu t + sigma^2 t^2 / 2)
Simpson intervals N=1000000

         t         K closed       K integral       |diff|   (K = log M)
   -0.3333      -0.11111111      -0.11111111     3.64e-13
   -0.1667      -0.11111111      -0.11111111     3.63e-13
  -0.03333     -0.031111111     -0.031111111     3.63e-13
         0                0   -3.6370906e-13     3.64e-13
   0.03333      0.035555556      0.035555556     3.64e-13
    0.1667       0.22222222       0.22222222     3.64e-13
       0.3             0.48             0.48     3.63e-13

moments from derivatives of K at t=0:
  mean  numeric 1  theory 1
  var   numeric 4  theory 4

pdf f(x)
     0.1991 |                                 ****                                 
     0.1859 |                               **    **                               
     0.1726 |                              *        *                              
     0.1593 |                             *          *                       

Exponential(2). M(t) only exists for t < lambda.

In [22]:
!cd moment_generating_functions && ./continuous_moment_generating_function exponential 2

Exponential(lambda=2)
M(t) = lambda / (lambda - t),  t < lambda
Simpson intervals N=1000000

         t         K closed       K integral       |diff|   (K = log M)
        -2      -0.69314718      -0.69314718     8.47e-13
        -1      -0.40546511      -0.40546511     8.47e-13
      -0.2      -0.09531018      -0.09531018     8.47e-13
         0                0    -8.467671e-13     8.47e-13
       0.2       0.10536052       0.10536052     8.47e-13
         1       0.69314718       0.69314718     8.47e-13
       1.8        2.3025851        2.3025851     8.47e-13

moments from derivatives of K at t=0:
  mean  numeric 0.50000017  theory 0.5
  var   numeric 0.25000013  theory 0.25

pdf f(x)
          2 |*                                                                     
      1.867 | *                                                                    
      1.734 |  *                                                                   
      1.601 |   *                                

Uniform(0, 2):

In [23]:
!cd moment_generating_functions && ./continuous_moment_generating_function uniform 0 2

Uniform(a=0, b=2)
M(t) = (e^{tb} - e^{ta}) / (t (b-a))
Simpson intervals N=1000000

         t         K closed       K integral       |diff|   (K = log M)
      -0.5      -0.45867515      -0.45867515     2.53e-14
     -0.25      -0.23960495      -0.23960495     5.07e-14
     -0.05     -0.049583368     -0.049583368        2e-15
         0                0                0            0
      0.05      0.050416632      0.050416632     1.22e-15
      0.25       0.26039505       0.26039505     1.65e-14
      0.45       0.48352507       0.48352507     3.06e-14

moments from derivatives of K at t=0:
  mean  numeric 1  theory 1
  var   numeric 0.33333333  theory 0.33333333

pdf f(x)
        0.5 |      **********************************************************      
     0.4667 |                                                                      
     0.4333 |                                                                      
        0.4 |                                                  

## Try it yourself

Change the numbers in a command above and re-run the cell. The arguments are described in the usage section at the top of this notebook.